In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 


df = pd.read_csv(r"C:\Users\AYAZ\.cache\kagglehub\datasets\yashdevladdha\uber-ride-analytics-dashboard\versions\2\uber_rides.csv")


In [3]:
#Cleaning
df['Booking ID']= df['Booking ID'].str.strip('"')
df['Customer ID']= df['Customer ID'].str.strip('"')

#Datetime merging
df['DateTime'] = pd.to_datetime(df['Date']+ ' ' + df['Time'])
df['Hour']= df['DateTime'].dt.hour
df['Month']=df['DateTime'].dt.month
df['DayofWeek']=df['DateTime'].dt.day_name()

In [4]:
#Subset Creation
df_completed = df[df['Booking Status'] == 'Completed'].copy()
df_cancelled_cust = df[df['Booking Status'] == 'Cancelled by Customer'].copy()
df_cancelled_driver = df[df['Booking Status']== 'Cancelled by Driver'].copy()

In [5]:
#Q1. What hour of day has the most rides and highest average fare?
hourly = df_completed.groupby('Hour')['Booking Value'].agg(['mean','count']).reset_index()
hourly.columns = ['Hour','Avg Fare','Ride Count']
hourly = hourly.sort_values(by='Ride Count', ascending=False)
hourly.head(5)


,Hour,Avg Fare,Ride Count
18,18,513.084810,7617
17,17,513.468513,6860
19,19,499.531921,6798
20,20,505.473605,6005
10,10,511.078389,5983


## Q1 - peak hours
Question: What hour of day has the most rides and highest average fare?
Insight : 6pm has the highest ride volume in the whole day. Average fare stays flat across all hours around (~₹505-519) suggesting no effective surge pricing.

In [6]:
#Q2. Which vehicle type generates the most revenue?
vehicle_revenue = df_completed.groupby('Vehicle Type')['Booking Value'].agg(['sum','mean','count']).reset_index()
vehicle_revenue.columns = ['Vehicle Type','Total Revenue','Avg Fare','Ride count']
vehicle_revenue = vehicle_revenue.sort_values(by='Total Revenue', ascending=False)
vehicle_revenue

,Vehicle Type,Total Revenue,Avg Fare,Ride count
0,Auto,11727615.0,506.483049,23155
2,Go Mini,9411418.0,507.381422,18549
3,Go Sedan,8538560.0,512.026865,16676
1,Bike,7144913.0,509.114508,14034
4,Premier Sedan,5733655.0,509.567632,11252
6,eBike,3298157.0,503.458556,6551
5,Uber XL,1406256.0,505.302192,2783


## Q2 - Vehicle Revenue
Question: Which vehicle type generates the most revenue?
Insight : Auto leads the charts with ₹11.7M driven by volume of (23k rides) not by price as all vehicles average ~₹506 per ride.


In [7]:
#Q3 cancellations reasons
cancel_reasons = df_cancelled_cust['Reason for cancelling by Customer'].value_counts().reset_index()
cancel_reasons.columns=['Reason','Count']
cancel_reasons = cancel_reasons.sort_values(by='Count', ascending=False)
print(cancel_reasons)

                                         Reason  Count
0                                 Wrong Address   2362
1                               Change of plans   2353
2  Driver is not moving towards pickup location   2335
3                        Driver asked to cancel   2295
4                             AC is not working   1155


## Q3- Cancellation Issues
Question: What is the most common reason customers cancel?
Insight: Top 4 reasons nearly equal at ~2300 each. 44% of customer cancellations are actually driver faults.

In [8]:
# Q4 Driver vs Customer Ratings Correlation
corr = df_completed[['Driver Ratings','Customer Rating']].corr()
print(corr)

                 Driver Ratings  Customer Rating
Driver Ratings          1.00000         -0.00101
Customer Rating        -0.00101          1.00000


## Q4 - Ratings
Question: Do higher rated drivers get better customer ratings?
Insight : Correlation of aroumd 0.001. Drivers and customers ratings are completely independent of each other mainly based on cutomer's mood.

In [9]:
# Q5 Payment method by vehicle type
ct = pd.crosstab(df_completed['Vehicle Type'], df_completed['Payment Method'])
print(ct)

Payment Method  Cash  Credit Card  Debit Card    UPI  Uber Wallet
Vehicle Type                                                     
Auto            5704         2325        1892  10358         2876
Bike            3501         1390        1130   6367         1646
Go Mini         4669         1838        1475   8409         2158
Go Sedan        4237         1715        1342   7405         1977
Premier Sedan   2739         1156         924   5058         1375
Uber XL          715          255         246   1218          349
eBike           1549          641         517   3019          825


## Q5 - Payment Method
Question: Which payment method dominates and does it vary by vehicle?
Insight : UPI dominates across all vehicle types and the payment behaviour is platform wide not vehicle specific